# Readout Guardrails: R1 / R2 / R3

This notebook adds readout diagnostics to ATR without presupposing outcome.

- **R1**: confidence-aware readout fields (margin, entropy)
- **R2**: ID-first token trace (IDs + display strings)
- **R3**: tensor-readout concordance categories

Interpretation principle:

> If token labels flicker while `cos_sim_mean` is high and margin is low, treat this as readout ambiguity first.


In [ ]:
# STEP 0: Imports and setup
import json
from pathlib import Path

import torch
from transformer_lens import HookedTransformer

from atr_engine import run_atr_loop

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# STEP 1: Configuration (neutral labels)
MODEL_NAME = "gpt2-small"
PROMPT = "The cat sat on the mat and then"
LAYER_START = 0
LAYER_END = 11
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

# Concordance thresholds for R3 categorisation
HIGH_COS_THRESHOLD = 0.995
LOW_MARGIN_THRESHOLD = 0.2

OUTPUT_PATH = Path("experiments/output/readout_guardrails_gpt2_small.json")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Layers:", LAYER_START, "->", LAYER_END)
print("Schedule:", ITERATION_SCHEDULE)

In [ ]:
# STEP 2: Run ATR loop with confidence-aware readout from atr_engine.py
model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()

with torch.no_grad():
    snapshots = run_atr_loop(
        model=model,
        prompt=PROMPT,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE,
        verbose=True,
    )

print(f"Snapshots captured: {len(snapshots)}")

In [ ]:
# STEP 3: R3 concordance classification (neutral categories)
def classify_snapshot(cos_sim_mean, top_logit_margin, high_cos, low_margin):
    if cos_sim_mean >= high_cos and top_logit_margin <= low_margin:
        return "high_cos_low_margin"
    if cos_sim_mean >= high_cos and top_logit_margin > low_margin:
        return "high_cos_high_margin"
    return "lower_cos"

summary_rows = []
category_counts = {
    "high_cos_low_margin": 0,
    "high_cos_high_margin": 0,
    "lower_cos": 0,
}

for s in snapshots:
    category = classify_snapshot(
        cos_sim_mean=float(s["cosine_sim_mean"]),
        top_logit_margin=float(s["top_logit_margin_last"]),
        high_cos=HIGH_COS_THRESHOLD,
        low_margin=LOW_MARGIN_THRESHOLD,
    )
    category_counts[category] += 1

    summary_rows.append({
        "iteration": int(s["iteration"]),
        "cosine_sim_mean": float(s["cosine_sim_mean"]),
        "cosine_sim_last": float(s["cosine_sim_last"]),
        "position_similarity": float(s["position_similarity"]),
        "top_token_id": int(s["top_token_ids_last"][0]),
        "top_token_string": str(s["top_token_strings_last"][0]),
        "top_token_prob": float(s["top_token_probs_last"][0]),
        "top_logit_margin": float(s["top_logit_margin_last"]),
        "entropy": float(s["entropy_last"]),
        "all_position_token_ids": [int(x) for x in s["all_position_token_ids"]],
        "all_position_token_strings": [str(x) for x in s["all_position_token_strings"]],
        "concordance_category": category,
    })

print("Category counts:", category_counts)

In [ ]:
# STEP 4: Save machine-readable output for follow-on analysis
payload = {
    "config": {
        "model_name": MODEL_NAME,
        "prompt": PROMPT,
        "layer_start": LAYER_START,
        "layer_end": LAYER_END,
        "max_iter": MAX_ITERATIONS,
        "schedule": ITERATION_SCHEDULE,
        "high_cos_threshold": HIGH_COS_THRESHOLD,
        "low_margin_threshold": LOW_MARGIN_THRESHOLD,
    },
    "category_counts": category_counts,
    "rows": summary_rows,
}

OUTPUT_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}")